# Building AI Applications with Snowflake Cortex
### RAG, Text-to-SQL & CoCo — Workshop Notebook

**Antes de executar o notebook**, complete as etapas de configuração via Worksheet SQL, de acordo com o  
guia do laboratório, para criar o database `AI_WORKSHOP_DB`, schemas, e o warehouse `WORKSHOP_WH`.

| Módulo | Tópico | Tempo |
|--------|-------|------|
| 1 | Your AI Foundation — Cortex Complete & RAG Concepts | 30 min |
| 2 | Production RAG with Cortex Search | 35 min |
| 3 | Text-to-SQL: From Naive to Production | 15 min |

## Setup — Imports & Session

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.core import Root
import pandas as pd
import time

session = get_active_session()
session.sql('USE DATABASE AI_WORKSHOP_DB').collect()
session.sql('USE SCHEMA RAG_DATA').collect()
session.sql('USE WAREHOUSE WORKSHOP_WH').collect()

def complete(model, prompt):
    escaped = prompt.replace("'", "''")
    return session.sql(
        f"SELECT SNOWFLAKE.CORTEX.COMPLETE('{model}', '{escaped}')"
    ).collect()[0][0]

Complete = complete  # alias used by RAG_App and Text-to-SQL cells

print('Session ready.')
print('  Database :', session.get_current_database())
print('  Schema   :', session.get_current_schema())
print('  Warehouse:', session.get_current_warehouse())

---
## Snowflake CoCo: Your AI Pair Programmer

Antes de iniciar o laboratório, use o **CoCo** (o ícone ✦ no seu Worksheet SQL)
para criar os objetos que o notebook precisa.

### Demo 1: Generate Setup SQL

No painel do CoCo, digite:

> *"Crie um internal stage no banco de dados AI_WORKSHOP_DB e no schema RAG_DATA chamado DOCS_STAGE com directory tables habilitadas e criptografia Snowflake SSE."*

O CoCo retornará:

```sql
CREATE OR REPLACE STAGE AI_WORKSHOP_DB.RAG_DATA.DOCS_STAGE
  DIRECTORY = (ENABLE = TRUE)
  ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE');
```

Aceite e execute. Você deve ver "Stage DOCS_STAGE criado."

### Demo 2: Explore an Unknown Function

Digite este prompt no CoCo:

> *"Como eu uso a função SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER para dividir o texto em blocos (chunks) de 1500 caracteres com uma sobreposição de 200 caracteres?"*

CoCo retornará um exemplo exibindo a sintaxe e parâmetos da função.

### Key Principle

> Use o CoCo para **gerar as partes complexas e entender o porquê** — não aceite o resultado cegamente. 
> O melhor fluxo é: descrever → gerar → ler → executar → iterar.

---

### Create DOCS_STAGE

A célula abaixo cria um internal stage chamado `DOCS_STAGE` — um local de armazenamento nomeado dentro do Snowflake onde os arquivos podem ser carregados. Se você concluiu a Demo 1 acima e já o criou via Worksheet SQL, esta célula não executará nada (`CREATE STAGE IF NOT EXISTS` skips creation if it already exists). **Execute-a de qualquer maneira antes de continuar.**

In [ ]:
# Safety net: if you skipped the CoCo demo in the SQL Worksheet, this creates the stage.
# If you already created it via CoCo, this is a no-op (CREATE STAGE IF NOT EXISTS).
session.sql("""
CREATE STAGE IF NOT EXISTS AI_WORKSHOP_DB.RAG_DATA.DOCS_STAGE
  DIRECTORY = (ENABLE = TRUE)
  ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')
""").collect()

print('Stage pronta: AI_WORKSHOP_DB.RAG_DATA.DOCS_STAGE')

---
## Module 1 — Your AI Foundation

| Pergunta | Tipo do Dado | Técnica |
|----------|-----------|----------|
| "O que a nossa política diz sobre reembolsos?" | Text, documents | **RAG** — retrieve relevant chunks, ground the LLM |
| "Qual foi o total de vendas do último trimestre?" | Tables | **Text-to-SQL** — translate to SQL, execute, return result |

### Step 1 — Your First LLM Call

`CORTEX.COMPLETE` chama os LLMs hospedados **dentro** da sua conta Snowflake —
sem API key, sem que os dados saiam do seu perímetro de segurança.

In [ ]:
model  = 'mistral-7b'
prompt = 'Em duas frases, explique a diferença entre dados estruturados e não estruturados.'
print(complete(model, prompt))

In [ ]:
for m in ['mistral-7b', 'llama3.1-70b']:
    print(f'--- {m} ---')
    print(complete(m, prompt))
    print()

### Step 2 — The Hallucination Problem

LLMs são treinados com dados públicos da internet. Pergunte algo privado ou recente, e eles inventam uma resposta que soa confiante.

In [ ]:
response = complete('mistral-7b', 'Responda no idioma português. Qual é o valor exato da receita da Snowflake no 4º trimestre do ano fiscal de 2026?')
print(response)
print()
print('O modelo supõe, usa respostas evasivas ou afirma algo que não pode ser confirmado — isso é alucinação.')

### Step 3 — RAG: Grounding Fixes Hallucination

Em vez de depender dos dados de treinamento, nós:
1. **Buscamos** fatos relevantes dos nossos próprios dados Snowflake
2. **Injetamos** esses fatos no prompt como contexto
3. **Instruimos** o modelo a responder usando apenas o que nós fornecemos

Usamos o `SNOWFLAKE_SAMPLE_DATA` — o conjunto de dados de pedidos do TPC-H, pré-carregado em todas as contas trial.
Isso mostra como pegar **dados ativos que já estão no Snowflake** e usá-los para embasar (ground) instantaneamente um LLM.

In [ ]:
top_customers = session.sql("""
    SELECT
        c.C_NAME,
        c.C_MKTSEGMENT,
        ROUND(SUM(o.O_TOTALPRICE), 0)       AS TOTAL_SPENT,
        COUNT(DISTINCT o.O_ORDERKEY)         AS NUM_ORDERS
    FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER c
    JOIN SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS   o ON c.C_CUSTKEY = o.O_CUSTKEY
    GROUP BY 1, 2
    ORDER BY TOTAL_SPENT DESC
    LIMIT 5
""").to_pandas()

lines = ['Os 5 principais clientes por valor total de pedidos (TPC-H sample database):']
for _, r in top_customers.iterrows():
    lines.append(
        f"  - {r['C_NAME']} | Segment: {r['C_MKTSEGMENT']} "
        f"| Revenue: ${r['TOTAL_SPENT']:,.0f} | Orders: {int(r['NUM_ORDERS'])}"
    )
data_context = '\n'.join(lines)

question = 'Quem é o principal cliente em revenue, em qual segmento ele está e quantos pedidos ele fez??'

grounded_prompt = (
    'Responda usando APENAS os dados abaixo. Não use conhecimento externo.\n\n'
    f'Data:\n{data_context}\n\n'
    f'Question: {question}\nAnswer:'
)

print('Live data from SNOWFLAKE_SAMPLE_DATA:')
print(data_context)
print()
print('Resposta "Grounded AI":')
print(complete('mistral-7b', grounded_prompt))

Em produção, o contexto não é criado manualmente — mas sim **recuperado automaticamente**
de um search index sobre todo o seu corpus de dados. É isso que o Module 2 constrói.

### Step 4 — Create Your Text Corpus

A célula abaixo carrega 15 descrições de recursos Snowflake —
a biblioteca de documentos que o seu aplicativo RAG vai pesquisar.

In [ ]:
session.sql("""
CREATE OR REPLACE TABLE AI_WORKSHOP_DB.RAG_DATA.FEATURE_DOCS (
    feature_name VARCHAR,
    content      VARCHAR
)
""").collect()

rows = [
    ('Cortex Complete', 'O Snowflake Cortex Complete fornece acesso aos principais modelos de linguagem de grande escala da indústria diretamente no Snowflake. Usando uma função SQL simples ou API Python, você pode interagir com LLMs sem mover os dados para fora do seu perímetro de segurança. Os modelos suportados incluem mistral-7b, mistral-large2, llama3.1-70b e snowflake-arctic.'),
    ('Cortex Search', 'O Cortex Search permite pesquisa e recuperação de alta qualidade e baixa latência em seus dados do Snowflake. Ele combina pesquisa de palavras-chave e vetor semântico e é otimizado para casos de uso de RAG. Você define um serviço de pesquisa sobre uma coluna de tabela e o Snowflake lida com a indexação e o embedding automaticamente.'),
    ('Cortex Analyst', 'O Cortex Analyst é um serviço de texto para SQL que permite aos usuários de negócios fazer perguntas em linguagem natural e receber respostas precisas baseadas em SQL. Ele usa um modelo semântico YAML para entender o esquema e a lógica de negócios. O Cortex Analyst está disponível via API REST e integrado ao Snowsight.'),
    ('Dynamic Tables', 'As Dynamic Tables permitem que você defina os resultados de uma consulta como uma tabela que é mantida atualizada automaticamente. Você escreve a lógica de transformação em SQL e o Snowflake gerencia o agendamento e as atualizações incrementais com base em um atraso (lag) desejado. Tanto a atualização completa quanto a incremental são suportadas.'),
    ('Snowpipe', "O Snowpipe é o serviço de ingestão contínua do Snowflake. Ele carrega dados de arquivos em staging assim que chegam, sem a necessidade de um warehouse em execução, usando um modelo de computação serverless. Pode ser acionado por notificações de armazenamento em nuvem ou chamadas de API REST."),
    ('Snowpark', "O Snowpark é o framework de desenvolvedor do Snowflake para código Python, Java e Scala que é executado diretamente na computação do Snowflake. Ele suporta pipelines de dados, treinamento de modelos de ML e UDFs sem mover os dados para fora do Snowflake."),
    ('Snowflake Notebooks', 'Os Snowflake Notebooks fornecem um ambiente de desenvolvimento interativo baseado em células no Snowsight. Eles suportam células Python, SQL e Markdown executadas na computação do Snowflake. Visualizações integradas e integrações de ML estão incluídas. Os Notebooks estão disponíveis em contas de teste (trial).'),
    ('Cortex Code', "O Cortex Code é o assistente de codificação com tecnologia de IA do Snowflake. Ele gera, explica e depura SQL e Python a partir de linguagem natural. Ele entende esquemas do Snowflake, APIs do Cortex e as melhores práticas, e está disponível como uma IDE autônoma e como Snowflake Copilot dentro do Snowsight em contas de teste (trial)."),
    ('Data Sharing', "O Snowflake Data Sharing permite que as organizações compartilhem dados dinâmicos e governados com outras contas do Snowflake sem movimentação de dados. Os provedores publicam no Marketplace ou compartilham diretamente com os consumidores, que consultam os dados com sua própria computação, sem custo de armazenamento para o provedor."),
    ('Snowflake Marketplace', 'O Snowflake Marketplace é uma plataforma de troca de dados para conjuntos de dados (datasets), serviços de dados e aplicativos nativos. Os consumidores acessam dados de terceiros diretamente no Snowflake sem ETL. O Marketplace inclui conjuntos de dados gratuitos e pagos em finanças, clima, saúde e muito mais.'),
    ('Cortex Search vs Cortex Analyst', 'O Cortex Search é para dados não estruturados, como documentos e texto, retornando passagens relevantes por meio de pesquisa híbrida. O Cortex Analyst é para tabelas estruturadas, gerando SQL a partir de linguagem natural. Ambos podem ser combinados para responder a perguntas em todos os tipos de dados.'),
    ('Task Graphs', 'Os Task Graphs orquestram vários procedimentos SQL ou Snowpark em uma ordem baseada em dependência. As tarefas são executadas em um cronograma, acionadas por um stream ou por uma tarefa pai. Os Task Graphs substituem orquestradores externos como o Airflow para pipelines nativos do Snowflake.'),
    ('Iceberg Tables', 'As Snowflake Iceberg Tables armazenam dados no formato Apache Iceberg em seu próprio armazenamento em nuvem, mantendo-se consultáveis por meio do Snowflake. Elas suportam DML completo e interoperabilidade com Spark e Trino, com os controles de governança e segurança do Snowflake.'),
    ('Streamlit in Snowflake', "O Streamlit no Snowflake permite construir e implantar aplicativos de dados interativos diretamente em sua conta Snowflake. Os aplicativos rodam na computação do Snowflake, consultam dados nativamente e são compartilháveis sem hospedagem externa. Disponível em contas de teste (trial)."),
    ('Snowflake AI Trust Layer', "O Snowflake AI Trust Layer garante que os dados processados pelos LLMs do Cortex nunca saiam da sua conta. A inferência do LLM ocorre dentro do perímetro de segurança do Snowflake, sem retenção de dados pelos provedores dos modelos, apoiando a conformidade com requisitos de privacidade e residência de dados.")
]

from snowflake.snowpark import Row
session.write_pandas(
    pd.DataFrame(rows, columns=['FEATURE_NAME', 'CONTENT']),
    table_name='FEATURE_DOCS',
    database='AI_WORKSHOP_DB',
    schema='RAG_DATA',
    overwrite=True
)

count = session.sql('SELECT COUNT(*) AS total FROM AI_WORKSHOP_DB.RAG_DATA.FEATURE_DOCS').collect()
print(f'Corpus carregados: {count[0]["TOTAL"]} documentos')
session.sql('SELECT feature_name, LEFT(content, 80) AS preview FROM FEATURE_DOCS').show()

---
## Module 2 — Production RAG with Cortex Search

```
FEATURE_DOCS  →  chunk text  →  CHUNKED_DOCS  →  Cortex Search Service  →  RAG_App
```

Execute cada célula em ordem. Cada etapa se baseia na anterior.

### Step 1 — Chunk the Text

Documentos reais são longos demais para caber em um único prompt de LLM. Nós os dividimos em chunks para que o search index possa recuperar o trecho mais relevante para uma determinada pergunta.

A célula abaixo define uma função `chunk_doc()` com dois parâmetros:
- `chunk_size=1500` — máximo de caracteres por chunk
- `overlap=200` — caracteres compartilhados entre chunks consecutivos (evita que as respostas sejam divididas no limite de um chunk)

Em seguida, ele lê `FEATURE_DOCS`, divide cada documento em chunks e escreve os resultados em `CHUNKED_DOCS`. Nossos documentos são curtos, então a maioria produz um único chunk cada. Com PDFs reais ou artigos longos, você veria muito mais chunks por documento de origem.

In [ ]:
def chunk_doc(text: str, chunk_size: int = 1500, overlap: int = 200) -> list:
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += chunk_size - overlap
    return chunks

docs_df = session.sql(
    'SELECT feature_name, content FROM AI_WORKSHOP_DB.RAG_DATA.FEATURE_DOCS'
).to_pandas()

rows = []
for _, row in docs_df.iterrows():
    for chunk in chunk_doc(row['CONTENT']):
        rows.append({'FEATURE_NAME': row['FEATURE_NAME'], 'CHUNK_TEXT': chunk})

session.write_pandas(
    pd.DataFrame(rows),
    table_name='CHUNKED_DOCS',
    database='AI_WORKSHOP_DB',
    schema='RAG_DATA',
    overwrite=True,
    auto_create_table=True
)

result = session.sql('SELECT COUNT(*) AS total FROM AI_WORKSHOP_DB.RAG_DATA.CHUNKED_DOCS').collect()
print(f'Chunks criados: {result[0]["TOTAL"]}')

### Step 2 — Create the Cortex Search Service

Uma única instrução DDL cria um search index híbrido totalmente gerenciado sobre `CHUNKED_DOCS`. O Snowflake lida com a geração de embeddings, a vetorização e a recuperação automaticamente — você apenas define qual coluna pesquisar e por quais atributos filtrar.

A célula abaixo cria o `FEATURE_SEARCH_SERVICE` e, em seguida, faz verificações periódicas até que o serviço esteja ativo (geralmente menos de 3 minutos). Você verá atualizações de status a cada 10 segundos. Aguarde a mensagem 'Service is ACTIVE' (O serviço está ATIVO) antes de prosseguir para o Step 3.

In [ ]:
session.sql("""
CREATE OR REPLACE CORTEX SEARCH SERVICE AI_WORKSHOP_DB.RAG_DATA.FEATURE_SEARCH_SERVICE
  ON chunk_text
  ATTRIBUTES feature_name
  WAREHOUSE       = WORKSHOP_WH
  TARGET_LAG      = '1 minute'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
  AS (
    SELECT chunk_text, feature_name
    FROM   AI_WORKSHOP_DB.RAG_DATA.CHUNKED_DOCS
  )
""").collect()

print('Service created. Waiting for initial indexing (up to 3 minutes)...')
for attempt in range(18):
    status_df  = session.sql('SHOW CORTEX SEARCH SERVICES IN SCHEMA AI_WORKSHOP_DB.RAG_DATA').to_pandas()
    status_str = ' '.join(status_df.astype(str).values.flatten()).upper()
    if 'ACTIVE' in status_str:
        print('Service is ACTIVE — ready to query.')
        break
    print(f'  Initializing... ({(attempt + 1) * 10}s elapsed)')
    time.sleep(10)
else:
    print('Still initializing. Proceed and re-run queries below if they fail.')

### Step 3 — Build the RAG Application

Nós construímos o `RAG_App` — uma classe unificada que lida com:
- **Unstructured queries** via Cortex Search (`app.query()`)
- **Structured data queries** via Cortex Analyst (`app.query_data()`) — usado no Module 3

> **Prompt CoCo para testar:**
> *"Escreva uma classe Python chamada RAG_App usando snowflake.core que consulta FEATURE_SEARCH_SERVICE
> em AI_WORKSHOP_DB.RAG_DATA, recupera os 3 principais chunks, e chama o Cortex Complete llama3.1-70b
> com um prompt grounded que instrui o modelo a responder a partir do contexto."*

Teste o prompt para ver o que o CoCo gera, em seguida **execute a célula abaixo** — a versão selecionada inclui o caminho de dados estruturados (`query_data()`) para o Module 3 que o CoCo não produzirá a partir deste prompt.

In [ ]:
import requests
import json

class RAG_App:
    def __init__(self, session, num_results: int = 3, semantic_view: str = None):
        self.session = session
        self.num_results = num_results
        self.semantic_view = semantic_view
        root = Root(session)
        self.svc = (
            root
            .databases['AI_WORKSHOP_DB']
            .schemas['RAG_DATA']
            .cortex_search_services['FEATURE_SEARCH_SERVICE']
        )

    # --- Unstructured path: Cortex Search + LLM ---
    def retrieve_context(self, question: str) -> list:
        resp = self.svc.search(query=question, limit=self.num_results)
        return [r['CHUNK_TEXT'] for r in resp.results]

    def generate_answer(self, question: str, context: list) -> str:
        ctx_str = '\n\n'.join(context)
        prompt = (
            'Você é um assistente de produtos da Snowflake.\n'
            'Responda APENAS usando o contexto abaixo. '
            'Se a resposta não estiver lá, diga "Eu não tenho essa informação."\n\n'
            f'Context:\n{ctx_str}\n\n'
            f'Question: {question}\nAnswer:'
        )
        return Complete('llama3.1-70b', prompt)

    def query(self, question: str) -> str:
        """Answer unstructured questions using Cortex Search + LLM."""
        context = self.retrieve_context(question)
        return self.generate_answer(question, context)

    # --- Structured path: Cortex Analyst + Semantic View (Module 3) ---
    def query_data(self, question: str):
        """Responda a perguntas sobre dados estruturados via Cortex Analyst.
        Retorne um DataFrame do pandas com os resultados da consulta."""
        if not self.semantic_view:
            raise ValueError("Nenhuma semantic_view configurada. Defina ao criar o RAG_App.")
        request_body = {
            "messages": [{"role": "user", "content": [{"type": "text", "text": question}]}],
            "semantic_view": self.semantic_view
        }
        conn = self.session.connection
        token = conn.rest.token
        host = conn.host
        resp = requests.post(
            f"https://{host}/api/v2/cortex/analyst/message",
            headers={"Authorization": f'Snowflake Token=\"{token}\"', "Content-Type": "application/json"},
            json=request_body
        )
        resp_content = resp.json()
        for content in resp_content["message"]["content"]:
            if content["type"] == "sql":
                return self.session.sql(content["statement"]).to_pandas()
        for content in resp_content["message"]["content"]:
            if content["type"] == "text":
                print(f"  Analyst: {content['text']}")
        return None


app = RAG_App(session)
print('RAG app ready.')
print('  app.query(question)      -> Cortex Search (unstructured docs)')
print('  app.query_data(question) -> Cortex Analyst (structured tables) [Module 3]')

### Step 4 — Test Your Application

In [ ]:
test_questions = [
    'Qual é a diferença entre o Cortex Search e o Cortex Analyst?',
    'Como o Snowpipe sabe quando novos arquivos chegam?',
    'Posso executar aplicativos Streamlit em uma conta trial?',
    'Meus dados saem do Snowflake quando eu chamo um LLM?',
    'O que é o Snowflake AI Trust Layer?',
]

for q in test_questions:
    print(f'Q: {q}')
    print(f'A: {app.query(q)}')
    print('-' * 60)

### Step 5 — Evaluate Response Quality

Nós usamos `CORTEX.COMPLETE` como um **LLM judge** — o mesmo modelo avalia cada resposta
em três métricas usando escalas de 0 a 1:

| Métrica | Pergunta |
|--------|----------|
| **Groundedness** | A resposta é suportada pelo contexto recuperado? |
| **Context Relevance** | O Cortex Search retornou chunks relevantes? |
| **Answer Relevance** | A resposta endereça à pergunta? |

In [ ]:
import re

def llm_score(prompt: str) -> float:
    result = Complete('llama3.1-70b', prompt).strip()
    nums   = re.findall(r'1\.0+|0\.\d+|[01]', result)
    return float(nums[0]) if nums else 0.0

def evaluate(question: str) -> dict:
    context = app.retrieve_context(question)
    answer  = app.generate_answer(question, context)
    ctx_str = '\n'.join(context)

    g = llm_score(
        f'Judge how grounded the answer is in the context.\n'
        f'1.0 = fully supported by context. 0.0 = not supported.\n'
        f'Return ONLY a number between 0.0 and 1.0.\n\n'
        f'Context: {ctx_str}\nAnswer: {answer}\nScore:'
    )
    cr = llm_score(
        f'Judge how relevant the context is for answering the question.\n'
        f'1.0 = directly useful. 0.0 = completely unrelated.\n'
        f'Return ONLY a number between 0.0 and 1.0.\n\n'
        f'Question: {question}\nContext: {ctx_str}\nScore:'
    )
    ar = llm_score(
        f'Judge how well the answer addresses the question.\n'
        f'1.0 = fully addresses it. 0.0 = completely irrelevant.\n'
        f'Return ONLY a number between 0.0 and 1.0.\n\n'
        f'Question: {question}\nAnswer: {answer}\nScore:'
    )
    return {'question': question, 'answer': answer,
            'groundedness': g, 'context_relevance': cr, 'answer_relevance': ar}


results = [evaluate(q) for q in test_questions]
eval_df = pd.DataFrame(results)

print(eval_df[['question','groundedness','context_relevance','answer_relevance']]
      .to_string(index=False))
print(f'\nMean — Groundedness: {eval_df["groundedness"].mean():.2f} | '
      f'Context Relevance: {eval_df["context_relevance"].mean():.2f} | '
      f'Answer Relevance: {eval_df["answer_relevance"].mean():.2f}')

---
## Module 3 — Text-to-SQL: From Naive to Production

No Module 2 nós construímos o `app.query()` para perguntas **unstructured** (documentos, PDFs).
Agora nós lidamos com dados **structured** — tabelas com números.

A abordagem naive: dar ao LLM o seu schema e deixá-llo escrever o SQL. Nós vamos provar que isso falhará, 
e depois consertar com uma **Semantic View** e com o **Cortex Analyst**.

| Step | O que fazer |
|------|-----------|
| 1 | Explorar o schema TPC-H |
| 2 | Estabelecer respostas gold-standard para Revenue e COGS |
| 3 | Deixar um naive LLM tentar — observe as falhas |
| 4 | Entender por que falhou |
| 5 | Criar uma Semantic View (com CoCo) |
| 6 | Usar Cortex Analyst para um acurácia em nível de produção |

### Step 1 — Explore the Schema

`SNOWFLAKE_SAMPLE_DATA` é pré-carregado em todas as contas trial Snowflake. Nós usamos o schema **TPC-H** —
um benchmark padrão que modela um distribuidor atacadista que compra peças de fornecedores e vende para clientes finais.

| Tabela | O que Representa | Linhas |
|-------|-------------------|------|
| **ORDERS** | Uma linha por pedido do cliente | 1.5M |
| **LINEITEM** | Uma linha por produto dentro de um pedido (1–7 por pedido) | 6M |
| **CUSTOMER** | Clientes | 150K |
| **PARTSUPP** | O que pagamos a cada fornecedor por peça | 800K |
| **NATION / REGION** | Tabelas de referência geográfica | 25 / 5 |

Key insight: `ORDERS.O_TOTALPRICE` **não** é a receita (revenue) — inclui impostos.
A verdadeira receita (revenue) deve ser calculada a partir de LINEITEM.

In [ ]:
schema_df = session.sql("""
    SELECT   table_name,
             TO_CHAR(row_count, '999,999,999') AS row_count
    FROM     SNOWFLAKE_SAMPLE_DATA.INFORMATION_SCHEMA.TABLES
    WHERE    table_schema = 'TPCH_SF1'
    ORDER BY table_name
""").to_pandas()

print('Tables in SNOWFLAKE_SAMPLE_DATA.TPCH_SF1:')
print(schema_df.to_string(index=False))

### Step 2 — Establish Gold-Standard Answers

Antes de perguntarmos qualquer coisa a um LLM, vamos estabelecer as fórmulas e os números **corretos**.

**Revenue** — valor líquido obtido com a venda de mercadorias (descontos aplicados, impostos excluídos):
```
Revenue = SUM( L_EXTENDEDPRICE x (1 - L_DISCOUNT) )
```

**Cost of Goods Sold (COGS)** — o que pagamos aos fornecedores pelas mercadorias que vendemos:
```
COGS = SUM( PS_SUPPLYCOST x L_QUANTITY )
```

Estes exigem tabelas diferentes: Revenue vem apenas da LINEITEM.
COGS exige fazer um join entre LINEITEM e PARTSUPP (supplier pricing).

Observe os valores do segmento **BUILDING** — nós os usaremos como nosso benchmark.

In [ ]:
print('REVENUE por Segmento de Mercado (Gold Standard)')
print('Fórmula: SUM(L_EXTENDEDPRICE * (1 - L_DISCOUNT))')
print()

correct_revenue = session.sql("""
    SELECT c.C_MKTSEGMENT AS segment,
           ROUND(SUM(l.L_EXTENDEDPRICE * (1 - l.L_DISCOUNT)), 0) AS revenue
    FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.LINEITEM l
    JOIN SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS o ON l.L_ORDERKEY = o.O_ORDERKEY
    JOIN SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER c ON o.O_CUSTKEY = c.C_CUSTKEY
    GROUP BY c.C_MKTSEGMENT ORDER BY revenue DESC
""").to_pandas()

for _, row in correct_revenue.iterrows():
    print(f"  {row['SEGMENT']:<12}  ${row['REVENUE']/1e9:>8.2f} B")

print()
print('COST OF GOODS SOLD por Segmento de Mercado (Gold Standard)')
print('Fórmula: SUM(PS_SUPPLYCOST * L_QUANTITY)')
print()

correct_cogs = session.sql("""
    SELECT c.C_MKTSEGMENT AS segment,
           ROUND(SUM(ps.PS_SUPPLYCOST * l.L_QUANTITY), 0) AS cogs
    FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.LINEITEM l
    JOIN SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS o ON l.L_ORDERKEY = o.O_ORDERKEY
    JOIN SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER c ON o.O_CUSTKEY = c.C_CUSTKEY
    JOIN SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.PARTSUPP ps
         ON l.L_PARTKEY = ps.PS_PARTKEY AND l.L_SUPPKEY = ps.PS_SUPPKEY
    GROUP BY c.C_MKTSEGMENT ORDER BY cogs DESC
""").to_pandas()

for _, row in correct_cogs.iterrows():
    print(f"  {row['SEGMENT']:<12}  ${row['COGS']/1e9:>8.2f} B")

print()
print('Benchmarks para lembrar:')
print('  BUILDING revenue ~ $44.1B')
print('  BUILDING COGS    ~ $15.5B')

### Step 3 — Naive Text-to-SQL: Let the LLM Try

Damos ao LLM o schema (nomes das tabelas e colunas) e pedimos para ele escrever o SQL.
Sem definições de negócio, sem fórmulas — apenas os nomes das colunas e uma pergunta.

In [ ]:
SCHEMA_CONTEXT = """
Tables (always use the full path SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.<table>):
- CUSTOMER(C_CUSTKEY, C_NAME, C_ADDRESS, C_NATIONKEY, C_PHONE, C_ACCTBAL, C_MKTSEGMENT)
- ORDERS(O_ORDERKEY, O_CUSTKEY, O_ORDERSTATUS, O_TOTALPRICE, O_ORDERDATE, O_ORDERPRIORITY)
- LINEITEM(L_ORDERKEY, L_PARTKEY, L_SUPPKEY, L_QUANTITY, L_EXTENDEDPRICE, L_DISCOUNT, L_TAX, L_SHIPDATE)
- NATION(N_NATIONKEY, N_NAME, N_REGIONKEY)
- REGION(R_REGIONKEY, R_NAME)
- PARTSUPP(PS_PARTKEY, PS_SUPPKEY, PS_AVAILQTY, PS_SUPPLYCOST)
- SUPPLIER(S_SUPPKEY, S_NAME, S_NATIONKEY, S_ACCTBAL)
"""

def ask_structured_data(question: str, show_sql: bool = True) -> pd.DataFrame:
    prompt = (
        'You are a Snowflake SQL expert.\n'
        'Write a single valid Snowflake SQL SELECT query that answers the question.\n'
        'Use only the tables in the schema. Return ONLY the SQL — no explanation, no markdown.\n\n'
        f'Schema:\n{SCHEMA_CONTEXT}\n\n'
        f'Question: {question}\nSQL:'
    )
    sql = Complete('llama3.1-70b', prompt).strip()
    sql = '\n'.join(l for l in sql.splitlines() if not l.strip().startswith('```')).strip()
    sql = sql.rstrip(';').strip()
    if show_sql:
        print(f'Generated SQL:\n{sql}\n')
    return session.sql(sql).limit(10).to_pandas()

# Test: Revenue by market segment
print('PERGUNTA: Qual é o revenue total por segmento de mercado?')
print()
llm_revenue = ask_structured_data('Qual é o revenue total por segmento de mercado?')
print('LLM Result:')
for _, row in llm_revenue.iterrows():
    print(f"  {row.iloc[0]:<12}  ${float(row.iloc[-1])/1e9:>8.2f} B")
print()
print('Compare com o gold standard de BUILDING: $44.14B')
print('O LLM usou o O_TOTALPRICE? Verifique o SQL acima.')

In [ ]:
# Note: this cell may take 1-3 minutes — LLM call uses mistral-large2 and the query joins LINEITEM (6M rows) to PARTSUPP (800K rows)
print('PERGUNTA: Qual é o COGS por segmento de mercado?')
print()

llm_cogs = None
try:
    llm_cogs = ask_structured_data('Qual é o COGS por segmento de mercado?')
    print('LLM Result:')
    for _, row in llm_cogs.iterrows():
        print(f"  {row.iloc[0]:<12}  ${float(row.iloc[-1])/1e9:>8.2f} B")
    print()
    print('Compare com o gold standard de BUILDING COGS: $15.49B')
    print('O LLM provavelmente calculou a RECEITA e a rotulou como COGS!')
except Exception as e:
    print(f'LLM SQL FALHOU: {e}')

In [ ]:
print('=' * 65)
print('REVENUE: Gold Standard vs Naive LLM')
print(f"  {'Segmento':<12} {'Correto':>12} {'LLM':>12} {'Erro':>8}")
print(f"  {'-'*12} {'-'*12} {'-'*12} {'-'*8}")

for _, row in correct_revenue.iterrows():
    seg = row['SEGMENT']
    correct_val = row['REVENUE']
    llm_row = llm_revenue[llm_revenue.iloc[:, 0] == seg]
    if not llm_row.empty:
        llm_val = float(llm_row.iloc[0].iloc[-1])
        pct = (llm_val - correct_val) / correct_val * 100
        print(f"  {seg:<12} ${correct_val/1e9:>8.2f}B  ${llm_val/1e9:>8.2f}B  {pct:>+5.1f}%")

print()
print('COGS: Gold Standard vs Naive LLM')

if llm_cogs is not None and not llm_cogs.empty:
    print(f"  {'Segmento':<12} {'Correto':>12} {'LLM':>12} {'Fator':>8}")
    print(f"  {'-'*12} {'-'*12} {'-'*12} {'-'*8}")
    for _, row in correct_cogs.iterrows():
        seg = row['SEGMENT']
        correct_val = row['COGS']
        llm_row = llm_cogs[llm_cogs.iloc[:, 0] == seg]
        if not llm_row.empty:
            llm_val = float(llm_row.iloc[0].iloc[-1])
            ratio = llm_val / correct_val
            print(f"  {seg:<12} ${correct_val/1e9:>8.2f}B  ${llm_val/1e9:>8.2f}B   {ratio:>4.1f}x")
else:
    print('  LLM COGS query failed entirely.')

print()
print('=' * 65)

### Step 4 — Why It Failed

Ambas as queries foram executadas sem erros. Ambas retornaram números que parecem plausíveis. Ambas estão **erradas**.

| Métrica | O que Aconteceu | Causa Raíz |
|--------|--------------|-----------|
| **Revenue** | Usou `SUM(O_TOTALPRICE)` em vez da fórmula do line-item | `O_TOTALPRICE` inclui impostos — um atalho conveniente que está sutilmente errado (+4%) |
| **COGS** | Calculou a receita e a rotulou como "cost" | Como não havia nenhuma coluna chamada "cost"ou "COGS" no nível correto, o LLM simplesmente pegou a fórmula de maior valor monetário que conseguiu encontrar (2.8x errado) |

O pattern: **Os LLMs tomam atalhos que não conseguem validar.** Quando há uma coluna com aparência plausível (`O_TOTALPRICE`), eles a pegam. Quando não há, eles improvisam — muitas vezes de forma equivocada.

Este é o risco em produção: **erros silenciosos que parecem corretos.**

### Step 5 — The Fix: Semantic Views

Uma **Semantic View** é um objeto do Snowflake que define formalmente o significado de negócio dos seus dados:

- **Metric definitions** — fórmulas exatas (e.g., revenue = `SUM(ext_price * (1 - discount))`)
- **Join paths** — quais tabelas se conectam a quais, e através de quais chaves
- **Synonyms** — para que "revenue", "sales", "net revenue" apontem todos para a mesma métrica
- **Dimensions** — atributos categóricos para agrupamento (segmento, região, ano)

Pense nisso como um **contrato entre sua lógica de negócio e a IA**.
É um objeto nativo do Snowflake com RBAC, compartilhamento, integração de catálogo e governança.

A célula abaixo cria a Semantic View de produção. Observe os campos `WITH SYNONYMS` e `COMMENT` — eles codificam o conhecimento de negócio que não pode ser inferido apenas pelos nomes das colunas. Execute a célula para criá-la.

In [ ]:
session.sql("USE SCHEMA ANALYTICS").collect()

create_sv_sql = """
CREATE OR REPLACE SEMANTIC VIEW AI_WORKSHOP_DB.ANALYTICS.TPCH_ORDER_ANALYTICS
  TABLES (
    line_items AS SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.LINEITEM PRIMARY KEY (L_ORDERKEY, L_LINENUMBER)
      COMMENT = 'Line items - one row per product per order',
    orders AS SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS PRIMARY KEY (O_ORDERKEY)
      COMMENT = 'Customer orders',
    customers AS SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER PRIMARY KEY (C_CUSTKEY)
      COMMENT = 'Customer master data',
    nations AS SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.NATION PRIMARY KEY (N_NATIONKEY)
      COMMENT = 'Nation reference',
    regions AS SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.REGION PRIMARY KEY (R_REGIONKEY)
      COMMENT = 'Region reference',
    partsupp AS SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.PARTSUPP PRIMARY KEY (PS_PARTKEY, PS_SUPPKEY)
      COMMENT = 'Part-supplier cost data'
  )
  RELATIONSHIPS (
    line_items_to_orders AS line_items (L_ORDERKEY) REFERENCES orders (O_ORDERKEY),
    orders_to_customers AS orders (O_CUSTKEY) REFERENCES customers (C_CUSTKEY),
    customers_to_nations AS customers (C_NATIONKEY) REFERENCES nations (N_NATIONKEY),
    nations_to_regions AS nations (N_REGIONKEY) REFERENCES regions (R_REGIONKEY),
    line_items_to_partsupp AS line_items (L_PARTKEY, L_SUPPKEY) REFERENCES partsupp (PS_PARTKEY, PS_SUPPKEY)
  )
  FACTS (
    line_items.discounted_price AS L_EXTENDEDPRICE * (1 - L_DISCOUNT)
      COMMENT = 'Net price after discount per line item',
    line_items.quantity AS L_QUANTITY
      COMMENT = 'Quantity ordered per line item',
    partsupp.supply_cost AS PS_SUPPLYCOST
      COMMENT = 'Unit cost we pay the supplier'
  )
  DIMENSIONS (
    orders.order_date AS O_ORDERDATE WITH SYNONYMS = ('order date', 'purchase date') COMMENT = 'Date placed',
    orders.order_year AS YEAR(O_ORDERDATE) COMMENT = 'Year placed',
    customers.customer_name AS C_NAME WITH SYNONYMS = ('customer name') COMMENT = 'Customer name',
    customers.market_segment AS C_MKTSEGMENT
      WITH SYNONYMS = ('market segment', 'segment')
      COMMENT = 'AUTOMOBILE, BUILDING, FURNITURE, HOUSEHOLD, MACHINERY',
    nations.nation_name AS N_NAME WITH SYNONYMS = ('country', 'nation') COMMENT = 'Nation name',
    regions.region_name AS R_NAME WITH SYNONYMS = ('region', 'continent', 'região')
      COMMENT = 'AFRICA, AMERICA, ASIA, EUROPE, MIDDLE EAST'
  )
  METRICS (
    line_items.total_revenue AS SUM(line_items.discounted_price)
      WITH SYNONYMS = ('revenue', 'net revenue', 'sales', 'receita')
      COMMENT = 'Total revenue: SUM(extended_price * (1 - discount))',
    line_items.total_cost_of_goods AS SUM(partsupp.supply_cost * line_items.quantity)
      WITH SYNONYMS = ('COGS', 'cost of goods sold', 'cost of goods', 'custo', 'CPV')
      COMMENT = 'Total cost of goods: SUM(supplier_cost * quantity)',
    line_items.total_profit AS SUM(line_items.discounted_price - partsupp.supply_cost * line_items.quantity)
      WITH SYNONYMS = ('profit', 'gross profit', 'margin', 'lucro', 'margem')
      COMMENT = 'Profit: revenue minus COGS',
    orders.order_count AS COUNT(DISTINCT O_ORDERKEY)
      WITH SYNONYMS = ('number of orders')
      COMMENT = 'Distinct order count',
    customers.customer_count AS COUNT(DISTINCT C_CUSTKEY)
      COMMENT = 'Distinct customer count'
  )
  COMMENT = 'Order analytics over TPC-H. Encodes revenue, COGS, and profit formulas.'
"""

result = session.sql(create_sv_sql).collect()
print('Semantic View criada: AI_WORKSHOP_DB.ANALYTICS.TPCH_ORDER_ANALYTICS')
print()
print('  Metrics encoded:')
print('    total_revenue       = SUM(L_EXTENDEDPRICE * (1 - L_DISCOUNT))')
print('    total_cost_of_goods = SUM(PS_SUPPLYCOST * L_QUANTITY)')
print('    total_profit        = revenue - COGS')

### Step 6 — Cortex Analyst: The Production Approach

**Cortex Analyst** é o serviço gerenciado de text-to-SQL Snowflake:

1. Lê as definições na **Semantic View** (metrics, dimensions, joins, synonyms)
2. Utiliza o state-of-the-art dos LLMs (selecionados automaticamente) para interpretar a pergunta
3. Gera um SQL que **segue exatamente as definições das suas métricas** — sem adivinhação
4. Utiliza o **Routing Mode** para gerar queries `SEMANTIC_VIEW()` para métricas governadas

Você não escolhe um LLM — o Cortex Analyst seleciona o melhor modelo para garantir a precisão.

A principal diferença: **a IA tem acesso à sua lógica de negócio**, não apenas aos nomes das colunas.

Já definimos o `query_data()` na nossa classe`RAG_App`. Agora, vamos conectá-lo à nossa semantic view.

In [ ]:
app = RAG_App(
    session,
    semantic_view='AI_WORKSHOP_DB.ANALYTICS.TPCH_ORDER_ANALYTICS'
)

def display_analyst_result(df, label):
    '''Display Cortex Analyst results, detecting column types automatically.'''
    if df is None or df.empty:
        print(f'  {label}: No results returned')
        return
    num_cols = df.select_dtypes(include='number').columns.tolist()
    str_cols = df.select_dtypes(include='object').columns.tolist()
    if num_cols and str_cols:
        dim_col, met_col = str_cols[0], num_cols[0]
        for _, row in df.iterrows():
            print(f"  {row[dim_col]:<12}  ${float(row[met_col])/1e9:>8.2f} B")
    else:
        print(df.to_string(index=False))

# Revenue via Cortex Analyst
print('app.query_data("Qual é o revenue total por segmento de mercado?")')
print()
ca_revenue = app.query_data("What is the total revenue by market segment?")
print('Resultado:')
display_analyst_result(ca_revenue, 'Revenue')

print()

# COGS via Cortex Analyst
print('app.query_data("Qual é o COGS por segmento de mercado?")')
print()
ca_cogs = app.query_data("What is the cost of goods sold by market segment?")
print('Resultado:')
display_analyst_result(ca_cogs, 'COGS')

print()
print('Ambos correspondem ao gold standard. Todas as vezes. De forma determinística.')
print('Porque o Cortex Analyst lê as definições da semantic view — ele não adivinha.')

### What We Proved

| | Naive LLM | Cortex Analyst + Semantic View |
|---|-----------|-------------------------------|
| **Revenue** | Usou `O_TOTALPRICE` — **+4% inflado** (incluiu impostos) | Fórmula correta todas as vezes |
| **COGS** | Calculou o revenue, rotulou como "cost" — **2.8x maior** | Fórmula correta todas as vezes |
| erros SQL? | Nenhum — ambas as consultas executaram sem problemas | Nenhum |
| Determinístico? | Não — varia entre as execuções | Sim — o mesmo SQL todas as vezes |

**O modo de falha do naive text-to-SQL não é 'a query falha'. É a query executa com sucesso com os números errados'."**

**A correção**: Codifique suas definições de negócio em uma Semantic View. Use o Cortex Analyst (`app.query_data()`) como a interface de produção.

Seu `RAG_App` agora suporta os dois:
- `app.query("O que é o Cortex Search?")` → resposta em texto a partir de documentos (Module 2)
- `app.query_data("Revenue por segmento?")` → DataFrame a partir de tabelas (Module 3)

In [ ]:
my_question = "Qual região tem a maior margem de lucro total?"

print(f'app.query_data("{my_question}")')
print()
result = app.query_data(my_question)
print('Result:')
print(result.to_string(index=False))

---
## Conclusion

### What You Built
- **Module 1:** Fez chamadas aos LLMs com o `CORTEX.COMPLETE`, demonstrou alucinação e aplicou a RAG grounding com dados em tempo real.
- **Module 2:** Pipeline RAG em produção — texto em chunks, `CORTEX SEARCH SERVICE`,
  classe `RAG_App` com busca e geração embasada (grounded)
- **Module 3:** Provou que o naive text-to-SQL falha silenciosamente (+4% na receita, 2.8x erro no COGS),
  depois construiu uma Semantic View e usou o Cortex Analyst para uma precisão de nível de produção

### Seu App suporta os dois Tipos de Dados
```python
app.query("O que é o Cortex Search?")       # Cortex Search (unstructured)
app.query_data("Receita por segmento?")     # Cortex Analyst (structured)
```

### Próximos Passos
| Idéia | Recurso |
|------|----------|
| Fazer o upload de PDFs reais com o `PARSE_DOCUMENT` | [Cortex Search docs](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview) |
| Adicionar uma interface de chat com Streamlit no Snowflake | [Streamlit docs](https://docs.snowflake.com/en/developer-guide/streamlit/about-streamlit) |
| Construir um agente unificado que roteia para o Search ou para o Analyst | [Cortex Agents](https://quickstarts.snowflake.com/guide/getting_started_with_cortex_agents) |